# Exercise 1: Digit Classification with Logistic Regression

In this notebook we will use simple logistic regression to see how far we can get when classifying images with digits on it. For this we use the famous MNIST dataset.

### 1. Data Loading and Analysis

We work with the MNIST dataset which is a large database of handwritten digits and very popular for testing machine learning models. You can find more details about the dataset [here](https://en.wikipedia.org/wiki/MNIST_database).

We download the dataset directly via the sklearn library. This may take a short while ... 

In [1]:
from sklearn.datasets import fetch_openml
x, y = fetch_openml('mnist_784', version=1, return_X_y=True, data_home="./data")

/usr/local/Caskroom/miniconda/base/envs/ml2/lib/python3.8/site-packages/sklearn/datasets/_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


Next, we have a brief look at the data:

In [2]:
print("X is a", type(x))
print("y is a", type(y))

X is a <class 'pandas.core.frame.DataFrame'>
y is a <class 'pandas.core.series.Series'>


In [3]:
x.shape

(70000, 784)

In [4]:
x.head(3)

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


One data point consists of 784 pixel values (=28x28 pixels) that represent an image showing a handwritten digit.
Let's look at the data for one data point:

In [5]:
x.iloc[0].describe()

count    784.000000
mean      35.108418
std       79.699674
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      255.000000
Name: 0, dtype: float64

The pixels of one datapoint have values between 0 and 255 which correspond to the grey value of that pixel. A value of zero means that the pixel is black, a value of 255 corresponds to a white pixel, every other value is some grey value in between.

The labels of the data are the digits that are actually shown on the image:

In [6]:
y.head(3)

0    5
1    0
2    4
Name: class, dtype: category
Categories (10, object): ['0', '1', '2', '3', ..., '6', '7', '8', '9']

In [7]:
type(y.iloc[0])

str

Since the label is of type string, we convert it to an integer:

In [8]:
y = y.astype("int")

To get an impression how the pictures of the digits look like, we visualize the first three rows with matplotlib. For this, we need to reshape each row to dimension 28x28 and then plot it.

import numpy as np
import matplotlib.pyplot as plt

digit = x.iloc[0]
digit_pixels = np.array(digit).reshape(28, 28)
plt.subplot(131)
plt.title("Label " + str(y.iloc[0]))
plt.imshow(digit_pixels, cmap=plt.cm.gray)
plt.axis('off')

digit = x.iloc[1]
digit_pixels = np.array(digit).reshape(28, 28)
plt.subplot(132)
plt.title("Label " + str(y.iloc[1]))
plt.imshow(digit_pixels, cmap=plt.cm.gray)
plt.axis('off')

digit = x.iloc[2]
digit_pixels = np.array(digit).reshape(28, 28)
plt.subplot(133)
plt.title("Label " + str(y.iloc[2]))
plt.imshow(digit_pixels, cmap=plt.cm.gray)
plt.axis('off')

Before we start with the model training we divide it into train and test data.

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

print("Train data points:", len(X_train))
print("Test data points:", len(X_test))

Train data points: 56000
Test data points: 14000


### 2. Model Training

Your exercise is now to train a logistic regression model using an one-vs-rest approach (see lecture slides).

For this, implement the following steps:
1. Apply min-max normalization to `X_train` and `X_test` to normalize the data.
2. Train a binary logistic regression to predict: "Is there the digit `9` on the image ?". Hint: For this you need to change the labels in `y_train`. Calculate the accuaracy of this classifer on the test data (you need to change also the labels in `y_test`).
3. Train a binary logistic regression on each digit from `0` to `9`. Save each model in a list.
4. For every test data point, predict which each model on it to find out which models returns the highest probability for the point. Hint: Use `.predict_proba()` to get class probabilities from each binary classifier.
5. Use the results from step 4 to calculate the accuracy of the overall approach on the test data. A correct implementation should achieve approximately 90% accuracy on test data.
7. Use `GridSearchCV` to tune the regularization parameter `C` (try values like 0.1, 1.0, 10.0) with 5-fold cross-validation.

In [10]:
#1 
# normalization: x' = (x - xmin) / (xmax - xmin) -> (x - 0) / (255 - 0) 
# x' = (x/255)

normX_train = X_train / 255
print(f"Min: {np.min(normX_train)}")
print(f"Max: {np.max(normX_train)}")

normX_test = X_test / 255
print(f"Min: {np.min(normX_test)}")
print(f"Max: {np.max(normX_test)}")

Min: 0.0
Max: 1.0
Min: 0.0
Max: 1.0


In [11]:
#2 
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=0, max_iter=1000)

print(y_test[:10])
yOneTrain = np.where(y_train == 9, 1, 0)
print(yOneTrain[:10])
yOneTest = np.where(y_test == 9, 1, 0)
print(yOneTest[:10])

model1 = model.fit(normX_train, yOneTrain)
model1.predict(normX_test)
model1.score(normX_test, yOneTest)

10840    0
56267    4
14849    1
62726    2
47180    7
61640    9
52730    7
21847    1
20394    1
45806    7
Name: class, dtype: int64
[0 1 0 0 0 0 1 0 0 1]
[0 0 0 0 0 1 0 0 0 0]


0.966

In [12]:
np.random.seed(0)
test = np.random.randint(0, 9, 20)
print(test)

res = np.where(test == 8, 1, 0)
print(res)

[5 0 3 3 7 3 5 2 4 7 6 8 8 1 6 7 7 8 1 5]
[0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0]


In [13]:
#3 
from sklearn.linear_model import LogisticRegression
modelList = []


def createModels() : 
   scoreList = []
   for i in range(10):
      model = LogisticRegression(random_state=0, max_iter=1000)
      y_trainData = np.where(y_train == i, 1, 0)
      y_testData = np.where(y_test == i, 1, 0)
      newModel = model.fit(normX_train, y_trainData)
      newModel.predict(normX_test)
      modelList.append(newModel)
      scoreList.append(newModel.score(normX_test, y_testData))
      print("Finished model " + f"{i}")
   print("ScoreList: ")
   print(scoreList)

createModels()
print(modelList)


Finished model 0
Finished model 1
Finished model 2
Finished model 3
Finished model 4
Finished model 5
Finished model 6
Finished model 7
Finished model 8
Finished model 9
ScoreList: 
[0.9930714285714286, 0.9916428571428572, 0.9787142857142858, 0.9746428571428571, 0.9834285714285714, 0.9754285714285714, 0.986, 0.9818571428571429, 0.9587857142857142, 0.966]
[LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0), LogisticRegression(max_iter=1000, random_state=0)]


In [14]:
#4
np.set_printoptions(suppress=True)
Scores = []
probabilities = []

for i in range(10): 
    y_testData = np.where(y_test == i, 1, 0)
    print("Probability of " + f"{i}" + " done")
    probabilities.append(modelList[i].predict_proba(normX_test)) 

print(np.shape(probabilities))
#print(probabilities)
    




Probability of 0 done
Probability of 1 done
Probability of 2 done
Probability of 3 done
Probability of 4 done
Probability of 5 done
Probability of 6 done
Probability of 7 done
Probability of 8 done
Probability of 9 done
(10, 14000, 2)


In [ ]:
np.set_printoptions(suppress=True)
results = np.array(probabilities)
print(results.shape)
results = results[:, :, 1]
temp = results.swapaxes(0, 1)
print(temp.shape)
predictions = np.argmax(temp, axis=1)
print(predictions.shape)
print(np.max(predictions))
print(np.min(predictions))


(10, 14000, 2)
(14000, 10)
(14000,)
9
0


In [16]:
#5
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, predictions))

0.9099285714285714


In [17]:
#6
from sklearn.model_selection import GridSearchCV

parameters = {'C':[0.6, 0.8, 1.0, 1.5, 2]}
#parameters = {'C':[0.1, 1.0, 10]}
baseModel = LogisticRegression(random_state=0, max_iter=1000)
gscv = GridSearchCV(baseModel, parameters, cv=5, verbose=3)
gscv.fit(normX_train, yOneTrain) 

Fitting 5 folds for each of 5 candidates, totalling 25 fits


[CV 1/5] END .............................C=0.6;, score=0.964 total time=  13.6s
[CV 2/5] END .............................C=0.6;, score=0.964 total time=  12.4s
[CV 3/5] END .............................C=0.6;, score=0.964 total time=  14.2s
[CV 4/5] END .............................C=0.6;, score=0.965 total time=  13.8s
[CV 5/5] END .............................C=0.6;, score=0.964 total time=  12.0s
[CV 1/5] END .............................C=0.8;, score=0.964 total time=  10.5s
[CV 2/5] END .............................C=0.8;, score=0.964 total time= 2.5min
[CV 3/5] END .............................C=0.8;, score=0.965 total time=  35.5s
[CV 4/5] END .............................C=0.8;, score=0.966 total time=  30.3s
[CV 5/5] END .............................C=0.8;, score=0.964 total time=  16.6s
[CV 1/5] END .............................C=1.0;, score=0.964 total time=  15.7s
[CV 2/5] END .............................C=1.0;, score=0.963 total time=  16.8s
[CV 3/5] END ...............

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000, random_state=0),
             param_grid={'C': [0.6, 0.8, 1.0, 1.5, 2]}, verbose=3)

In [18]:
import pandas as pd
resultsDf = pd.DataFrame(gscv.cv_results_)
resultsDf
print(gscv.best_params_)
print(gscv.best_score_)

{'C': 0.8}
0.9644107142857143
